In [19]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/plot.py'>

In [20]:
IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
    
IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

In [21]:
IN_pair = 'G140S-Q148H'
# IN_pair = 'Y143C-S230R'
# IN_pair = 'G140A-Q148K'
# IN_pair = 'G140S-Q148R'
# IN_pair = 'G140S-Q148K'

# IN_pair = 'Y143R-N155H'
pair1, pair2 = functions.split_pairs(IN_pair)

p1_reduced = functions.unreduced_to_reduced(IN_redux, pair1)
p2_reduced = functions.unreduced_to_reduced(IN_redux, pair2)

wt1,pos1,mt1 = functions.split_pair(p1_reduced)
wt2,pos2,mt2 = functions.split_pair(p2_reduced)


p_SH = functions.calculate_double_mutant_probablity(IN_consensus_seq,p1_reduced,p2_reduced,IN_J,1,263)
print(p_SH)

0.13951657972904513


In [22]:
# IN_pair = 'G140S-Q148H'
IN_pair = 'Y143C-S230R'
# IN_pair = 'G140A-Q148K'
# IN_pair = 'G140S-Q148R'
# IN_pair = 'G140S-Q148K'

# IN_pair = 'Y143R-N155H'

min_pos = 1
max_pos = 263
pair1, pair2 = functions.split_pairs(IN_pair)

p1_reduced = functions.unreduced_to_reduced(IN_redux, pair1)
p2_reduced = functions.unreduced_to_reduced(IN_redux, pair2)

wt1,pos1,mt1 = functions.split_pair(p1_reduced)
wt2,pos2,mt2 = functions.split_pair(p2_reduced)

gof_seqs = []
gof_without_DMC = []
gof_with_DMC = []

rescue_seqs = []

noncomp_seqs = []


consensus_result = functions.calculate_delta_delta_e(p1_reduced, p2_reduced, IN_consensus_seq, IN_J, min_pos, max_pos)
consensus_flip_relation = consensus_result[0] - consensus_result[1]

for IN_seq in IN_all_seq:
    # Debugging: Print inputs to the function

    # result_wom = functions.calculate_delta_delta_e(p1_reduced, p2_reduced, IN_seq, IN_J, 1, 263)
    # result_wm = functions.calculate_with_double_mutation_dde(p1_reduced, p2_reduced, IN_seq, IN_J, 1, 263)

    result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, IN_seq, IN_J, 1, 263)

    # Check if the result is None

    # if result_wom is None:
    #     continue 
    # pair1_de, pair2_de, pair12_de, pair12_dde = result_wom

    # if result_wm is None:
    #     continue 
    # pair1_de, pair2_de, pair12_de, pair12_dde = result_wm

    if result_merged is None:
        continue 
    pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

    if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de >0:
        #gain of function
        gof_seqs.append(IN_seq)
        if IN_seq[pos1-min_pos] == wt1 and IN_seq[pos2-min_pos] == wt2:
            gof_without_DMC.append(IN_seq)
        
        if IN_seq[pos1-min_pos] == mt1 and IN_seq[pos2-min_pos] == mt2:
            gof_with_DMC.append(IN_seq)
        
    elif pair1_de < pair12_de and pair2_de < pair12_de:
        #rescue
        rescue_seqs.append(IN_seq)
        
    elif pair1_de < pair12_de or pair2_de < pair12_de:
        #compensatory
        continue
    else:
        #non-compensatory
        noncomp_seqs.append(IN_seq)


#####################
print(len(gof_seqs), "GoF sequences found.")
p_SH_values = []
for gof_seq in gof_seqs:
    p_SH = functions.calculate_double_mutant_probablity(gof_seq,p1_reduced,p2_reduced,IN_J,1,263)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for GoF sequences:", average_p_SH)

######################
print(len(rescue_seqs), "rescue sequences found.")
p_SH_values = []
for rescue_seq in rescue_seqs:
    p_SH = functions.calculate_double_mutant_probablity(rescue_seq,p1_reduced,p2_reduced,IN_J,1,263)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for GoF sequences:", average_p_SH)

#######################
print(len(noncomp_seqs), "non-compensatory sequences found.")
p_SH_values = []
for noncomp_seq in noncomp_seqs:
    p_SH = functions.calculate_double_mutant_probablity(noncomp_seq,p1_reduced,p2_reduced,IN_J,1,263)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for GoF sequences:", average_p_SH)



25 GoF sequences found.
Average p_SH for GoF sequences: 0.5839335497471736
241 rescue sequences found.
Average p_SH for GoF sequences: 0.026767761335144663
589 non-compensatory sequences found.
Average p_SH for GoF sequences: 0.00014522519683262834


In [ ]:
# PR_pair = 'D30N-N88D'
# PR_pair = 'V32I-I47V'
# PR_pair = 'G48V-I54A'
PR_pair = 'D30N-K45Q'
# PR_pair = 'I54A-V82A'

min_pos = 1
max_pos = 99
pair1, pair2 = functions.split_pairs(PR_pair)

p1_reduced = functions.unreduced_to_reduced(PR_redux, pair1)
p2_reduced = functions.unreduced_to_reduced(PR_redux, pair2)

wt1, pos1, mt1 = functions.split_pair(p1_reduced)
wt2, pos2, mt2 = functions.split_pair(p2_reduced)

gof_seqs = []
gof_without_DMC = []
gof_with_DMC = []

rescue_seqs = []

noncomp_seqs = []

consensus_result = functions.calculate_delta_delta_e(p1_reduced, p2_reduced, PR_consensus_seq, PR_J, min_pos, max_pos)
consensus_flip_relation = consensus_result[0] - consensus_result[1]

for PR_seq in PR_all_seq:
    result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, PR_seq, PR_J, min_pos, max_pos)

    if result_merged is None:
        continue
    pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

    if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
        # gain of function
        gof_seqs.append(PR_seq)
        if PR_seq[pos1 - min_pos] == wt1 and PR_seq[pos2 - min_pos] == wt2:
            gof_without_DMC.append(PR_seq)

        if PR_seq[pos1 - min_pos] == mt1 and PR_seq[pos2 - min_pos] == mt2:
            gof_with_DMC.append(PR_seq)

    elif pair1_de < pair12_de and pair2_de < pair12_de:
        # rescue
        rescue_seqs.append(PR_seq)

    elif pair1_de < pair12_de or pair2_de < pair12_de:
        # compensatory
        continue
    else:
        # non-compensatory
        noncomp_seqs.append(PR_seq)

#####################
print(len(gof_seqs), "GoF sequences found.")
p_SH_values = []
for gof_seq in gof_seqs:
    p_SH = functions.calculate_double_mutant_probablity(gof_seq, p1_reduced, p2_reduced, PR_J, min_pos, max_pos)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for GoF sequences:", average_p_SH)

######################
print(len(rescue_seqs), "rescue sequences found.")
p_SH_values = []
for rescue_seq in rescue_seqs:
    p_SH = functions.calculate_double_mutant_probablity(rescue_seq, p1_reduced, p2_reduced, PR_J, min_pos, max_pos)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for rescue sequences:", average_p_SH)

#######################
print(len(noncomp_seqs), "non-compensatory sequences found.")
p_SH_values = []
for noncomp_seq in noncomp_seqs:
    p_SH = functions.calculate_double_mutant_probablity(noncomp_seq, p1_reduced, p2_reduced, PR_J, min_pos, max_pos)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for non-compensatory sequences:", average_p_SH)

117 GoF sequences found.
Average p_SH for GoF sequences: 0.5945075070021693
923 rescue sequences found.
Average p_SH for rescue sequences: 0.12256968846902588
1924 non-compensatory sequences found.
Average p_SH for non-compensatory sequences: 0.0007638448486228985


In [ ]:
RT_pair = 'F116Y-Q151M'
# RT_pair = 'M41L-T215Y'

min_pos = 39
max_pos = 226
pair1, pair2 = functions.split_pairs(RT_pair)

p1_reduced = functions.unreduced_to_reduced(RT_redux, pair1)
p2_reduced = functions.unreduced_to_reduced(RT_redux, pair2)

wt1, pos1, mt1 = functions.split_pair(p1_reduced)
wt2, pos2, mt2 = functions.split_pair(p2_reduced)

gof_seqs = []
gof_without_DMC = []
gof_with_DMC = []

rescue_seqs = []

noncomp_seqs = []

consensus_result = functions.calculate_delta_delta_e(p1_reduced, p2_reduced, RT_consensus_seq, RT_J, min_pos, max_pos)
consensus_flip_relation = consensus_result[0] - consensus_result[1]

for RT_seq in RT_all_seq:
    result_merged = functions.calculate_dde_v2(p1_reduced, p2_reduced, RT_seq, RT_J, min_pos, max_pos)

    if result_merged is None:
        continue
    pair1_de, pair2_de, pair12_de, pair12_dde = result_merged

    if pair1_de < pair12_de and pair2_de < pair12_de and pair12_de > 0:
        # gain of function
        gof_seqs.append(RT_seq)
        if RT_seq[pos1 - min_pos] == wt1 and RT_seq[pos2 - min_pos] == wt2:
            gof_without_DMC.append(RT_seq)

        if RT_seq[pos1 - min_pos] == mt1 and RT_seq[pos2 - min_pos] == mt2:
            gof_with_DMC.append(RT_seq)

    elif pair1_de < pair12_de and pair2_de < pair12_de:
        # rescue
        rescue_seqs.append(RT_seq)

    elif pair1_de < pair12_de or pair2_de < pair12_de:
        # compensatory
        continue
    else:
        # non-compensatory
        noncomp_seqs.append(RT_seq)

#####################
print(len(gof_seqs), "GoF sequences found.")
p_SH_values = []
for gof_seq in gof_seqs:
    p_SH = functions.calculate_double_mutant_probablity(gof_seq, p1_reduced, p2_reduced, RT_J, min_pos, max_pos)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for GoF sequences:", average_p_SH)

######################
print(len(rescue_seqs), "rescue sequences found.")
p_SH_values = []
for rescue_seq in rescue_seqs:
    p_SH = functions.calculate_double_mutant_probablity(rescue_seq, p1_reduced, p2_reduced, RT_J, min_pos, max_pos)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for rescue sequences:", average_p_SH)

#######################
print(len(noncomp_seqs), "non-compensatory sequences found.")
p_SH_values = []
for noncomp_seq in noncomp_seqs:
    p_SH = functions.calculate_double_mutant_probablity(noncomp_seq, p1_reduced, p2_reduced, RT_J, min_pos, max_pos)
    p_SH_values.append(p_SH)

average_p_SH = sum(p_SH_values) / len(p_SH_values) if p_SH_values else 0
print("Average p_SH for non-compensatory sequences:", average_p_SH)

6991 GoF sequences found.
Average p_SH for GoF sequences: 0.7523727185273982
3386 rescue sequences found.
Average p_SH for rescue sequences: 0.12947043626555507
2264 non-compensatory sequences found.
Average p_SH for non-compensatory sequences: 0.006840089685053662
